In [ ]:
# OPTIONAL: Kiểm tra nhanh runtime device.
# Notebook 00 là ingestion/manifest notebook nên không bắt buộc có GPU.
try:
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Đang sử dụng thiết bị: {device}")
except ImportError:
    print("torch chưa được cài trong runtime này. Bỏ qua kiểm tra device; notebook 00 không cần GPU.")


## Harness smoke readiness markers

GitHub repo setup uses `git clone` on first run and `git pull` on later runs, then installs System 1 with `pip install -e`.

Runtime roots used by local, Colab, and Kaggle execution: `AIC_REPO_PARENT`, `AIC_REPO_ROOT`, `AIC_DATA_ROOT`, `AIC_RUNTIME_ROOT`, `AIC_ARTIFACT_ROOT`, and `AIC_HF_REPO_ID`.

Notebook 00 phase00 handoff uses `sync-phase00-ingestion` after HF raw ingest and batch assignment complete.

# Notebook 00B — Drive shadow + streaming HF raw upload + phase00 upload

Phiên bản B dùng workflow streaming để giảm DriveFS read/write và tránh tràn local storage trên Colab free CPU:

```text
Google Drive folder của BTC / nguồn tổ chức
→ drive-shadow copy sang folder Drive của team/bạn
→ remount Drive và kiểm tra Colab nhìn thấy đủ zip
→ stream-standardize-upload-raw:
   scan zip members để lập pairing plan
   extract/probe các video/json theo batch nhỏ theo `RAW_UPLOAD_BATCH_SIZE` trong `/content/aic_scratch`
   upload mỗi batch lên Hugging Face raw repo bằng batched commit
   dùng `--min-free-gb`, `--drive-sync-sleep-seconds`, `--cleanup-every-files`, `--cleanup-every-gb` để giữ local scratch an toàn
   ghi progress/report nhỏ
   cleanup scratch ngay sau từng batch
→ kiểm tra HF raw repo
→ ingest từ HF raw repo để tạo processed artifacts local
→ assign-batches tạo batch_manifest.csv + batch_*.txt
→ copy audit reports vào local release manifests
→ upload phase00 ingestion artifacts lên Hugging Face processed repo bằng `sync-phase00-ingestion`
→ kiểm tra HF processed repo theo `phase00_ingestion` layout
```

Thiết kế repo:

```text
AIC26_raw     = versioned canonical raw store
AIC26_release = processed workspace + final release repo
```

Raw repo dùng version prefix:

```text
AIC26_raw/
└── canonical_raw_v009/
    ├── raw_videos/
    ├── metadata/
    ├── frame_timeline/{video_id}.parquet
    └── manifests/
        ├── canonical_file_manifest.jsonl
        ├── canonical_import_report.json
        ├── canonical_video_inventory.parquet
        ├── missing_metadata.json
        └── unmatched_metadata.json
```

Processed repo dùng phase layout theo release prefix:

```text
AIC26_release/
└── canonical_release_v009/
    ├── phase00_ingestion/
    │   ├── tables/videos.parquet
    │   ├── raw_mapping/media_store_manifest.parquet
    │   ├── frame_timeline/{video_id}.parquet
    │   ├── manifests/frame_timeline_manifest.parquet
    │   ├── manifests/batch_manifest.csv
    │   ├── manifests/batch_*.txt
    │   └── reports/
    │       ├── dataset_report.json
    │       ├── ingestion_errors.jsonl
    │       ├── missing_metadata.json
    │       ├── unmatched_metadata.json
    │       ├── drive_shadow_report.json
    │       ├── canonical_import_report.json
    │       └── stream_standardize_upload_progress_{raw_import_id}.jsonl
    ├── phase01_structure/
    ├── phase02_features/
    ├── phase03_merged/
    ├── releases/
    ├── checkpoints/
    └── logs/
```

`phase00_ingestion` là output của Notebook 00, chưa phải final runtime release. Final app-ready release cho System 2 nằm trong `AIC26_release/canonical_release_vXXX/releases/competition_dataset_vXXX/`.

Legacy flat layout `canonical_release_vXXX/{manifests,tables,raw_mapping,frame_timeline}` đã deprecated; output mới phải dùng `canonical_release_vXXX/phase00_ingestion/{manifests,tables,raw_mapping,frame_timeline,reports}`.

Điểm quan trọng:

- `drive_target_id` phải là **folder ID đúng của folder local `archive_source_dir`**. Ví dụ nếu `archive_source_dir = /content/drive/MyDrive/AIC2026/raw_dataset`, thì `drive_target_id` phải chính là folder ID của `raw_dataset` trên Google Drive.
- `scratch_dir` **phải nằm trên local runtime** để tránh ghi temp qua Google DriveFS. Không đặt scratch trong `/content/drive`.
- Không tạo full `standardize/raw_videos` và `standardize/metadata` trên Drive trong workflow B.
- `missing_metadata.json` và `unmatched_metadata.json` là raw-level audit manifests ở `AIC26_raw/canonical_raw_vXXX/manifests/`. `AIC26_release/canonical_release_vXXX/phase00_ingestion/reports/` giữ snapshot/copy cho từng run.
- Production dùng `frame_timeline_policy=required`: stream upload tạo `frame_timeline/{video_id}.parquet` cho mọi video; HF ingest tải Parquet này thay vì tải lại MP4 để probe.
- HF ingest dùng `--canonical-hf-repo-id` và `--canonical-hf-prefix`.
- Notebook chỉ orchestration/verification. Logic storage contract chính nằm trong package CLI.


In [ ]:
import os
import sys
from pathlib import Path
from dataclasses import dataclass

@dataclass
class WorkflowConfig:
    # 1. Hugging Face repos
    # Raw repo: versioned canonical raw store.
    hf_canonical_repo: str = "1thesudden/AIC26_raw"
    raw_import_id: str = "canonical_raw_v009"

    # Processed repo: release/control plane.
    hf_release_repo: str = "1thesudden/AIC26_release"
    release_id: str = "canonical_release_v009"

    # Stream zip media/metadata qua local scratch rồi upload lên HF raw repo.
    # Bật True khi muốn chạy workflow HF raw → HF ingest.
    run_upload_standardized_raw: bool = True

    # 2. Google Drive IDs
    # Workflow bắt buộc: copy source Drive folder của BTC sang folder Drive của team/bạn trước.
    drive_source_id: str = "1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK" # https://drive.google.com/drive/folders/1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK?usp=drive_link
    drive_target_id: str = "1zV5neo0APq9PmOmc66PmfGayFngczaWw" # https://drive.google.com/drive/folders/1zV5neo0APq9PmOmc66PmfGayFngczaWw?usp=drive_link
    run_drive_shadow: bool = True

    # 3. Local paths nhìn từ Colab sau khi mount Google Drive.
    # RẤT QUAN TRỌNG: archive_source_dir phải là local mount path tương ứng với drive_target_id.
    # drive-shadow copy vào drive_target_id; standardize đọc từ archive_source_dir.
    archive_source_dir: str = "/content/drive/MyDrive/AIC2026/raw_dataset/video_batch1"
    archive_target_dir: str = "/content/drive/MyDrive/AIC2026/raw_dataset/stream_reports"

    # Scratch phải nằm trên local runtime. Workflow B extract/probe theo batch nhỏ rồi cleanup ngay.
    # Không đặt scratch/temp trong /content/drive cho dataset lớn.
    scratch_dir: str = "/content/aic_scratch"
    stream_progress_path: str = f"{archive_target_dir}/stream_standardize_upload_progress_{raw_import_id}.jsonl"
    # 4. Repo code
    github_repo_url: str = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
    github_branch: str = "system1-notebook01"
    repo_dir_name: str = "Multimodal-Agentic-Retrieval-Engine"

    # 5. Workflow options
    package_mode: str = "bronze_fast"
    num_batches: int = 10
    frame_timeline_policy: str = "required"
    timeline_workers: str = "auto"
    ingest_max_workers: int = 4
    min_free_gb: float = 15.0
    drive_sync_sleep_seconds: int = 30
    cleanup_every_files: int = 100
    cleanup_every_gb: float = 50.0

    run_ingest: bool = True
    run_assign_batches: bool = True

    # Upload processed artifacts nhỏ sau khi chia batch.
    run_upload_processed: bool = True

    def __post_init__(self):
        self.env = "colab" if "google.colab" in sys.modules else "local"
        self.workspace = Path("/content") if self.env == "colab" else Path.cwd()
        os.environ["AIC_RELEASE_ID"] = self.release_id
        os.environ["AIC_HF_REPO_ID"] = self.hf_release_repo

config = WorkflowConfig()

try:
    timeline_cpu_count = len(os.sched_getaffinity(0))
except (AttributeError, OSError):
    timeline_cpu_count = os.cpu_count() or 1
timeline_workers_preview = max(1, min(2, timeline_cpu_count)) if config.timeline_workers == "auto" else int(config.timeline_workers)

print("Môi trường:", config.env)
print("Workspace:", config.workspace)
print("Release ID:", config.release_id)
print("HF processed repo:", config.hf_release_repo)
print("HF raw repo:", config.hf_canonical_repo)
print("Raw import ID:", config.raw_import_id)
print("Run upload standardized raw:", config.run_upload_standardized_raw)
print("Run drive shadow:", config.run_drive_shadow)
print("Drive source ID:", config.drive_source_id)
print("Drive target ID:", config.drive_target_id)
print("Archive source dir dùng để standardize:", config.archive_source_dir)
print("Stream report dir:", config.archive_target_dir)
print("Scratch dir:", config.scratch_dir)
print("Stream progress path:", config.stream_progress_path)
print("Frame timeline policy:", config.frame_timeline_policy)
print("Timeline workers requested/resolved:", config.timeline_workers, timeline_workers_preview, "CPU:", timeline_cpu_count)
print("Ingest max workers:", config.ingest_max_workers)


In [ ]:
# BƯỚC 1: Mount Google Drive, authenticate Google, và lấy HF token.
import os
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Mount Drive và lấy HF token.
if config.env == "colab":
    from google.colab import userdata, drive, auth

    hf_token = userdata.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN trong Colab Secrets. Hãy thêm HF_TOKEN trước khi chạy notebook.")

    os.environ["HF_TOKEN"] = hf_token
    os.environ["AIC_HF_TOKEN"] = hf_token

    # Giữ output Colab sạch hơn khi dùng huggingface_hub trong subprocess.
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    os.environ["HF_HUB_VERBOSITY"] = "error"
    os.environ.setdefault("AIC_VERBOSE", "0")
    os.environ.setdefault("AIC_HF_PROGRESS", "0")

    drive.mount("/content/drive", force_remount=False)
    auth.authenticate_user()
else:
    if not os.environ.get("HF_TOKEN") and not os.environ.get("AIC_HF_TOKEN"):
        print("Cảnh báo: chưa thấy HF_TOKEN/AIC_HF_TOKEN trong environment.")

    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    os.environ["HF_HUB_VERBOSITY"] = "error"
    os.environ.setdefault("AIC_VERBOSE", "0")
    os.environ.setdefault("AIC_HF_PROGRESS", "0")


In [ ]:
# BƯỚC 2: Clone hoặc sync repo code.
repo_dir = Path(config.workspace) / config.repo_dir_name
repo_dir = repo_dir.expanduser().resolve()

# Quan trọng: nếu kernel đang đứng trong repo cũ đã bị xóa, phải cd về thư mục còn tồn tại.
os.chdir("/content")

print("Current cwd:", Path.cwd())
print("repo_dir:", repo_dir)
print("repo_dir exists:", repo_dir.exists())
print("repo_dir .git exists:", (repo_dir / ".git").exists())
print("github_repo_url:", config.github_repo_url)
print("github_branch:", config.github_branch)

def run_git(cmd, *, cwd=None, check=True):
    safe_cwd = Path(cwd).expanduser().resolve() if cwd else Path("/content")
    safe_cwd.mkdir(parents=True, exist_ok=True)

    print("CWD:", safe_cwd)
    print("RUN:", " ".join(map(str, cmd)))

    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(safe_cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if result.stdout:
        print(result.stdout)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit_code={result.returncode}\n"
            f"CMD: {' '.join(map(str, cmd))}\n\n"
            f"OUTPUT:\n{result.stdout}"
        )

    return result

if (repo_dir / ".git").exists():
    print("Repo đã tồn tại, cập nhật an toàn:", repo_dir)
    dirty = run_git(["git", "status", "--porcelain"], cwd=repo_dir, check=False)
    if dirty.stdout.strip():
        raise RuntimeError("Repo đang có thay đổi local. Hãy commit/stash thủ công trước khi chạy notebook; notebook không reset hoặc xóa thay đổi.")
    run_git(["git", "fetch", "origin", "--prune"], cwd=repo_dir)
    remote_branch = f"origin/{config.github_branch}"
    branch_check = run_git(["git", "rev-parse", "--verify", remote_branch], cwd=repo_dir, check=False)
    if branch_check.returncode != 0:
        raise RuntimeError(f"Không tìm thấy remote branch {remote_branch}.")
    local_branch = run_git(["git", "rev-parse", "--verify", f"refs/heads/{config.github_branch}"], cwd=repo_dir, check=False)
    if local_branch.returncode == 0:
        run_git(["git", "switch", config.github_branch], cwd=repo_dir)
    else:
        run_git(["git", "switch", "--track", "-c", config.github_branch, remote_branch], cwd=repo_dir)
    run_git(["git", "merge", "--ff-only", remote_branch], cwd=repo_dir)
else:
    if repo_dir.exists():
        raise RuntimeError(f"repo_dir đã tồn tại nhưng không phải Git repo: {repo_dir}. Chọn path khác hoặc xử lý thủ công.")
    print("Clone repo:", config.github_repo_url)
    run_git(["git", "clone", "--branch", config.github_branch, "--single-branch", config.github_repo_url, str(repo_dir)], cwd="/content")

print("Git commit hiện tại:")
run_git(["git", "log", "-1", "--oneline"], cwd=repo_dir)

print("Git status:")
run_git(["git", "status", "--short"], cwd=repo_dir)

In [ ]:
# 2B. Kiểm tra project_root và setup import path cho notebook kernel.
import sys
import subprocess
from pathlib import Path

print("\nCheck package source layout:")

project_root_candidates = [
    repo_dir / "system1",
    repo_dir,
]

project_root = None
for candidate in project_root_candidates:
    print("- candidate:", candidate)
    print("  has src/system1:", (candidate / "src" / "system1").exists())
    print("  has src/system1/__init__.py:", (candidate / "src" / "system1" / "__init__.py").exists())
    print("  has runtime/environment.py:", (candidate / "src" / "system1" / "runtime" / "environment.py").exists())

    if (candidate / "src" / "system1" / "__init__.py").exists():
        project_root = candidate.resolve()
        break

if project_root is None:
    raise RuntimeError(
        "Không tìm thấy project_root chứa src/system1/__init__.py.\n"
        "Checked:\n" + "\n".join(str(p) for p in project_root_candidates)
    )

system1_src = project_root / "src"

print("\nInstalling editable package from:", project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root)],
    check=True,
)

# Đưa src path thật lên đầu sys.path để notebook import đúng package.
system1_src_str = str(system1_src)
sys.path = [system1_src_str] + [
    p for p in sys.path
    if str(Path(p).expanduser().resolve()) != system1_src_str
]

# Xóa cache nếu trước đó import nhầm system1 thành namespace package.
for module_name in list(sys.modules):
    if module_name == "system1" or module_name.startswith("system1."):
        del sys.modules[module_name]

import system1

REPO_ROOT = repo_dir
SYSTEM1_ROOT = project_root

print("\nResolved project paths:")
print("- REPO_ROOT:", REPO_ROOT)
print("- SYSTEM1_ROOT:", SYSTEM1_ROOT)
print("- system1_src:", system1_src)
print("- system1 file:", getattr(system1, "__file__", None))
print("- system1 path:", list(getattr(system1, "__path__", [])))

environment_module_path = system1_src / "system1" / "runtime" / "environment.py"
print("- environment.py exists:", environment_module_path.exists())

if getattr(system1, "__file__", None) is None:
    raise RuntimeError(
        "system1 vẫn đang bị import thành namespace package. "
        "Hãy restart runtime rồi chạy lại từ BƯỚC 1."
    )

system1_file = Path(system1.__file__).resolve()
expected_package_dir = (system1_src / "system1").resolve()
if system1_file.parent != expected_package_dir:
    raise RuntimeError(
        "Kernel đang import system1 từ source khác repo vừa sync: "
        f"actual={system1_file}, expected_under={expected_package_dir}. "
        "Hãy restart runtime rồi chạy lại từ BƯỚC 1."
    )
print("- package source preflight: OK")

In [ ]:
# BƯỚC 3: Định nghĩa helper run_cli để gọi system1 CLI trong notebook.
def run_cli(args, *, check=True, stream=True, tail_lines=300):
    import os
    import subprocess
    import sys
    from pathlib import Path

    def find_repo_root():
        candidates = [
            globals().get("SYSTEM1_ROOT"),
            globals().get("REPO_ROOT"),
            Path.cwd(),
            Path("/content/Multimodal-Agentic-Retrieval-Engine"),
        ]

        for candidate in candidates:
            if candidate is None:
                continue
            path = Path(candidate).expanduser().resolve()

            # Nếu đang ở project package system1/.
            if path.name == "system1" and (path / "src" / "system1").exists():
                return path

            # Nếu đang ở repo root có folder system1/src/system1.
            if (path / "system1" / "src" / "system1").exists():
                return path

        raise RuntimeError(
            "Không tìm thấy repo root. Hãy chạy cell clone/setup repo trước, "
            "hoặc kiểm tra repo có nằm ở /content/Multimodal-Agentic-Retrieval-Engine không."
        )

    cli_cwd = find_repo_root()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    # Giữ tương thích với bản cũ:
    # - nếu cli_cwd là repo root cha: dùng system1/src
    # - nếu cli_cwd là package root system1/: dùng src
    if (cli_cwd / "src" / "system1").exists():
        system1_src = cli_cwd / "src"
    else:
        system1_src = cli_cwd / "system1" / "src"

    env["PYTHONPATH"] = str(system1_src) + os.pathsep + env.get("PYTHONPATH", "")

    # Chặn progress bar nội bộ của huggingface_hub trong subprocess.
    # Nếu cần debug HF progress bar thật, set AIC_HF_PROGRESS=1 trước khi gọi run_cli.
    if env.get("AIC_HF_PROGRESS", "0") != "1":
        env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        env["HF_HUB_VERBOSITY"] = "error"

    # Mặc định không in per-file/per-cleanup nếu package đã hỗ trợ AIC_VERBOSE.
    env.setdefault("AIC_VERBOSE", "0")

    cmd = [sys.executable, "-m", "system1.cli", *args]

    print("\n" + "=" * 100)
    print("CWD:", cli_cwd)
    print("PYTHONPATH prefix:", system1_src)
    print("RUN:", " ".join(cmd))
    print("stream:", stream)
    print("=" * 100)

    if not stream:
        completed = subprocess.run(
            cmd,
            cwd=str(cli_cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

        output = completed.stdout or ""
        output_lines = output.splitlines()
        tail = "\n".join(output_lines[-tail_lines:])

        print(f"CLI finished: exit_code={completed.returncode}")
        print(f"Last {min(tail_lines, len(output_lines))} lines:")
        print("-" * 100)
        print(tail)
        print("-" * 100)

        if check and completed.returncode != 0:
            raise RuntimeError(
                f"CLI failed with exit code {completed.returncode}: {' '.join(args)}\n\n"
                f"Last output:\n{tail}"
            )

        return output

    process = subprocess.Popen(
        cmd,
        cwd=str(cli_cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    returncode = process.wait()
    output = "".join(lines)

    if check and returncode != 0:
        tail = "\n".join(output.splitlines()[-tail_lines:])
        raise RuntimeError(
            f"CLI failed with exit code {returncode}: {' '.join(args)}\n\n"
            f"Last output:\n{tail}"
        )

    return output

In [ ]:
# BƯỚC 4: Resolve runtime paths và kiểm tra mapping Drive workflow.
from system1.runtime.environment import resolve_runtime_paths
from pathlib import Path

runtime_paths = resolve_runtime_paths(output_root=config.workspace / "output", release_id=config.release_id)
output_base = runtime_paths.output_root
release_root = output_base / config.release_id
output_base.mkdir(parents=True, exist_ok=True)

print("Runtime paths:")
print("- environment:", runtime_paths.environment)
print("- workspace_root:", runtime_paths.workspace_root)
print("- input_root:", runtime_paths.input_root)
print("- output_root:", runtime_paths.output_root)
print("- artifact_root:", runtime_paths.artifact_root)
print("- release_id:", runtime_paths.release_id)

archive_source = Path(config.archive_source_dir)
archive_target = Path(config.archive_target_dir)
print("Workflow check:")
print("- drive-shadow sẽ copy vào drive_target_id:", config.drive_target_id)
print("- standardize sẽ đọc archive_source_dir sau khi Drive được mount:", archive_source)
print("- Hai giá trị này phải trỏ tới cùng một folder Google Drive.")

print("Trạng thái archive_source_dir trước drive-shadow:")
print("- exists:", archive_source.exists())
if archive_source.exists():
    visible_items = list(archive_source.rglob("*"))
    print("- visible item count before shadow:", len(visible_items))
    for p in visible_items[:30]:
        print(" ", p)
else:
    print("- Chưa thấy folder local. Notebook vẫn sẽ chạy drive-shadow trước, sau đó remount Drive và kiểm tra lại.")

In [ ]:
# BƯỚC 5: Drive-shadow copy từ folder BTC/source sang folder Drive của team/bạn.
from pathlib import Path
import json
import time
from google.colab import drive

drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"
archive_source = Path(config.archive_source_dir)

if config.run_drive_shadow:
    if not config.drive_source_id or not config.drive_target_id:
        raise RuntimeError("run_drive_shadow=True nhưng thiếu drive_source_id hoặc drive_target_id.")

    print("Bắt đầu drive-shadow:")
    print("- source_folder_id:", config.drive_source_id)
    print("- dest_folder_id:", config.drive_target_id)
    print("- report_path:", drive_shadow_report_path)

    run_cli([
        "drive-shadow",
        "--source-folder-id", config.drive_source_id,
        "--dest-folder-id", config.drive_target_id,
        "--report-path", str(drive_shadow_report_path),
    ])
else:
    raise RuntimeError(
        "Workflow hiện tại yêu cầu chạy drive-shadow trước. "
        "Hãy để config.run_drive_shadow = True."
    )

In [ ]:
# BƯỚC 6: Kiểm tra report của drive-shadow trước khi đọc dữ liệu từ Drive mount.
from pathlib import Path
import json

drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

print("Kiểm tra drive-shadow report:")
print("- path:", drive_shadow_report_path)
print("- exists:", drive_shadow_report_path.exists())

drive_shadow_report = None
copied_files = 0
created_folders = 0
skipped_existing = 0
skipped_google_apps = 0
error_count = None

if drive_shadow_report_path.exists():
    drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))

    copied_files = int(drive_shadow_report.get("copied_files", 0))
    created_folders = int(drive_shadow_report.get("created_folders", 0))
    skipped_existing = int(drive_shadow_report.get("skipped_existing", 0))
    skipped_google_apps = int(drive_shadow_report.get("skipped_google_apps", 0))
    error_count = int(drive_shadow_report.get("error_count", 0))

    print("Drive shadow summary:")
    print("- status:", drive_shadow_report.get("status"))
    print("- source_folder_id:", drive_shadow_report.get("source_folder_id"))
    print("- dest_folder_id:", drive_shadow_report.get("dest_folder_id"))
    print("- source_folder_name:", drive_shadow_report.get("source_folder_name"))
    print("- dest_folder_name:", drive_shadow_report.get("dest_folder_name"))
    print("- copied_files:", copied_files)
    print("- created_folders:", created_folders)
    print("- skipped_existing:", skipped_existing)
    print("- skipped_google_apps:", skipped_google_apps)
    print("- error_count:", error_count)

    if error_count and error_count > 0:
        print("\nMột số item lỗi đầu tiên:")
        failed_items = [
            item for item in drive_shadow_report.get("items", [])
            if item.get("status") == "failed"
        ]
        for item in failed_items[:20]:
            print(item)

        raise RuntimeError(
            "drive-shadow có error_count > 0. "
            "Không chạy standardize khi copy Drive chưa sạch lỗi."
        )

    if copied_files + created_folders + skipped_existing + skipped_google_apps == 0:
        raise RuntimeError(
            "drive-shadow chạy xong nhưng report cho thấy không copy/tạo/skip bất kỳ item nào. "
            "Kiểm tra source folder hoặc quyền Drive."
        )
else:
    raise FileNotFoundError(
        f"Không thấy drive-shadow report: {drive_shadow_report_path}. "
        "CLI drive-shadow cần tạo report sau khi chạy."
    )

In [ ]:
# BƯỚC 7: Remount Google Drive và xác nhận Colab đã nhìn thấy đủ file.
from pathlib import Path
import json
import time
from google.colab import drive

archive_source = Path(config.archive_source_dir)
drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

print("Remount Google Drive để Colab nhìn thấy dữ liệu mới...")

try:
    drive.flush_and_unmount()
except Exception as exc:
    print("flush_and_unmount warning:", exc)

time.sleep(5)
drive.mount("/content/drive", force_remount=True)

print("Kiểm tra local archive_source_dir sau drive-shadow:")
print("- archive_source_dir:", archive_source)
print("- exists:", archive_source.exists())

if not archive_source.exists():
    raise RuntimeError(
        "Không thấy archive_source_dir sau drive-shadow. "
        "archive_source_dir phải là local path tương ứng với drive_target_id."
    )

if not drive_shadow_report_path.exists():
    raise RuntimeError(f"Không thấy drive-shadow report để đối chiếu: {drive_shadow_report_path}")

drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))

# Chỉ các file thường đã copied/skipped_existing mới cần xuất hiện trong Drive mount.
# skipped_google_apps không phải file local; created_folders không tính vào file count.
expected_relative_files = sorted({
    str(item.get("path", "")).strip("/")
    for item in drive_shadow_report.get("items", [])
    if item.get("kind") == "file" and item.get("status") in {"copied", "skipped_existing"} and item.get("path")
})

expected_file_count_from_report = int(drive_shadow_report.get("copied_files", 0)) + int(drive_shadow_report.get("skipped_existing", 0))
expected_file_count = len(expected_relative_files) or expected_file_count_from_report

print("Drive mount sync expectation:")
print("- expected_file_count:", expected_file_count)
print("- expected_relative_files sample:", expected_relative_files[:20])

visible_files = []
visible_relative_files = set()
missing_expected = set(expected_relative_files)

# LỖI CŨ: cell dừng ngay khi thấy visible_items > 0, ví dụ mới thấy 4/15 file.
# SỬA: chỉ pass khi thấy đủ file theo drive_shadow_report.
for attempt in range(1, 31):
    visible_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
    visible_relative_files = {
        p.relative_to(archive_source).as_posix()
        for p in visible_files
    }

    if expected_relative_files:
        missing_expected = set(expected_relative_files) - visible_relative_files
        ready = len(missing_expected) == 0
    else:
        missing_expected = set()
        ready = len(visible_files) >= expected_file_count if expected_file_count else len(visible_files) > 0

    print(
        f"- attempt {attempt}/30: "
        f"visible_file_count={len(visible_files)} "
        f"expected_file_count={expected_file_count} "
        f"missing_expected={len(missing_expected)}"
    )

    if ready:
        break

    if missing_expected:
        print("  missing sample:", sorted(missing_expected)[:10])

    time.sleep(10)

if expected_file_count and len(visible_files) < expected_file_count:
    raise RuntimeError(
        f"Drive mount mới thấy {len(visible_files)}/{expected_file_count} file sau drive-shadow. "
        "Không chạy standardize vì sẽ chỉ xử lý một phần dataset. "
        "Đây thường là lỗi sync/cache của Google Drive mount trong Colab. "
        "Hãy đợi thêm vài phút rồi chạy lại riêng cell kiểm tra này."
    )

if expected_relative_files and missing_expected:
    raise RuntimeError(
        f"Drive mount vẫn thiếu {len(missing_expected)} file theo drive-shadow report. "
        f"Ví dụ thiếu: {sorted(missing_expected)[:20]}. "
        "Không chạy standardize khi local mount chưa thấy đủ file."
    )

print("Local Drive mount đã thấy đủ dữ liệu theo drive-shadow report:")
print("- visible_file_count:", len(visible_files))
for p in visible_files[:50]:
    print(" ", p)


In [ ]:
# BƯỚC 8: Stream standardize zip pairs và upload HF raw.
# Workflow B không tạo full raw_videos/metadata trên Drive.
# Package scan zip members để lập pairing plan, extract/probe theo batch nhỏ theo RAW_UPLOAD_BATCH_SIZE trong local scratch, upload HF bằng batched commit, ghi progress rồi cleanup scratch.

from pathlib import Path
import json
import shutil
import subprocess

archive_source = Path(config.archive_source_dir)
stream_report_dir = Path(config.archive_target_dir)
scratch_dir = Path(config.scratch_dir)
stream_progress_path = Path(config.stream_progress_path)
drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

print("BƯỚC 8: Stream standardize + upload raw to HF")
print("- archive_source:", archive_source)
print("- scratch_dir:", scratch_dir)
print("- stream_report_dir:", stream_report_dir)
print("- stream_progress_path:", stream_progress_path)
print("- hf_canonical_repo:", config.hf_canonical_repo)
print("- raw_import_id:", config.raw_import_id)

if not archive_source.exists():
    raise RuntimeError(f"Không thấy archive_source_dir: {archive_source}")

scratch_str = str(scratch_dir.resolve() if scratch_dir.exists() else scratch_dir)
if scratch_str.startswith("/content/drive/"):
    raise RuntimeError(
        f"scratch_dir đang nằm trên Google Drive mount: {scratch_dir}. "
        "Workflow B cần scratch local, ví dụ /content/aic_scratch."
    )
if not scratch_str.startswith("/content/"):
    print("WARNING: Trên Colab nên dùng scratch_dir dưới /content để cleanup nhanh.")

stream_report_dir.mkdir(parents=True, exist_ok=True)
stream_progress_path.parent.mkdir(parents=True, exist_ok=True)

legacy_progress_path = stream_report_dir / "stream_standardize_upload_progress.jsonl"
if not stream_progress_path.exists() and legacy_progress_path.exists():
    latest_by_video = {}
    for line in legacy_progress_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        inventory = record.get("inventory") if isinstance(record.get("inventory"), dict) else {}
        record_repo_id = record.get("raw_repo_id") or inventory.get("canonical_repo_id")
        record_import_id = record.get("raw_import_id") or inventory.get("canonical_prefix")
        video_id = record.get("video_id")
        if record_repo_id == config.hf_canonical_repo and record_import_id == config.raw_import_id and video_id:
            latest_by_video[str(video_id)] = record
    if latest_by_video:
        stream_progress_path.write_text(
            "".join(json.dumps(record, sort_keys=True, ensure_ascii=False) + "\n" for record in latest_by_video.values()),
            encoding="utf-8",
        )
        print(f"Migrated {len(latest_by_video)} latest records từ legacy progress: {legacy_progress_path}")

# Cleanup scratch trước khi chạy; chỉ xóa local scratch, không xóa Drive source/progress.
if scratch_dir.exists():
    print("Cleanup local scratch trước khi stream:", scratch_dir)
    shutil.rmtree(scratch_dir, ignore_errors=True)
scratch_dir.mkdir(parents=True, exist_ok=True)

print("Disk trước stream upload:")
subprocess.run("df -h / /content /content/drive 2>/dev/null || df -h / /content", shell=True, check=False)
subprocess.run(f"du -sh {str(scratch_dir)!r} 2>/dev/null || true", shell=True, check=False)

source_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
zip_files = [p for p in source_files if p.suffix.lower() == ".zip"]
print("- source file count:", len(source_files))
print("- zip file count:", len(zip_files))
for p in zip_files[:30]:
    print(" zip:", p)

if not zip_files:
    raise RuntimeError("archive_source_dir không có zip file để stream.")

# Guard chống lỗi Drive mount chỉ thấy một phần file sau drive-shadow.
if drive_shadow_report_path.exists():
    drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))
    expected_relative_files = sorted({
        str(item.get("path", "")).strip("/")
        for item in drive_shadow_report.get("items", [])
        if item.get("kind") == "file" and item.get("status") in {"copied", "skipped_existing"} and item.get("path")
    })
    expected_file_count = len(expected_relative_files) or (
        int(drive_shadow_report.get("copied_files", 0)) + int(drive_shadow_report.get("skipped_existing", 0))
    )
    if expected_relative_files:
        source_relative_files = {p.relative_to(archive_source).as_posix() for p in source_files}
        missing_expected = sorted(set(expected_relative_files) - source_relative_files)
        if missing_expected:
            raise RuntimeError(
                f"archive_source_dir còn thiếu {len(missing_expected)} file theo drive-shadow report. "
                f"Ví dụ thiếu: {missing_expected[:10]}"
            )
    elif expected_file_count and len(source_files) < expected_file_count:
        raise RuntimeError(
            f"archive_source_dir chỉ thấy {len(source_files)}/{expected_file_count} file theo drive-shadow report."
        )

# Không parse output `--help`: Typer/Rich có thể rút gọn tên option theo độ rộng terminal.
# run_cli luôn ép PYTHONPATH về source vừa sync; lần gọi command thật bên dưới là contract check chính xác.

if config.run_upload_standardized_raw:
    run_cli([
        "stream-standardize-upload-raw",
        "--source-dir", str(archive_source),
        "--target-hf-repo-id", config.hf_canonical_repo,
        "--raw-import-id", config.raw_import_id,
        "--scratch-dir", str(scratch_dir),
        "--progress-path", str(stream_progress_path),
        "--resume",
        "--no-overwrite",
        "--min-free-gb", str(config.min_free_gb),
        "--drive-sync-sleep-seconds", str(config.drive_sync_sleep_seconds),
        "--cleanup-every-files", str(config.cleanup_every_files),
        "--cleanup-every-gb", str(config.cleanup_every_gb),
        "--frame-timeline-policy", config.frame_timeline_policy,
        "--timeline-workers", config.timeline_workers,
    ], stream=True, tail_lines=500)
else:
    print("config.run_upload_standardized_raw=False, skip stream upload raw.")

print("Disk sau stream upload:")
subprocess.run("df -h / /content /content/drive 2>/dev/null || df -h / /content", shell=True, check=False)
subprocess.run(f"du -sh {str(scratch_dir)!r} 2>/dev/null || true", shell=True, check=False)

if stream_progress_path.exists():
    lines = stream_progress_path.read_text(encoding="utf-8").splitlines()
    print("stream progress records:", len(lines))
    print("stream progress path:", stream_progress_path)
else:
    print("WARNING: stream progress path chưa tồn tại:", stream_progress_path)


In [ ]:
# BƯỚC 9: Kiểm tra progress local sau stream upload.
# Raw repo check đầy đủ nằm ở bước 10. Cell này chỉ xác nhận checkpoint/resume file nhỏ trên Drive.

from pathlib import Path
import json

stream_progress_path = Path(config.stream_progress_path)
scratch_dir = Path(config.scratch_dir)

print("BƯỚC 9: Check stream progress")
print("- stream_progress_path:", stream_progress_path)
print("- scratch_dir:", scratch_dir)

if config.run_upload_standardized_raw:
    if not stream_progress_path.exists():
        raise RuntimeError(f"Không thấy stream progress JSONL: {stream_progress_path}")
    records = [json.loads(line) for line in stream_progress_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    latest_by_video = {}
    for record in records:
        inventory = record.get("inventory") if isinstance(record.get("inventory"), dict) else {}
        record_repo_id = record.get("raw_repo_id") or inventory.get("canonical_repo_id")
        record_import_id = record.get("raw_import_id") or inventory.get("canonical_prefix")
        video_id = record.get("video_id")
        if record_repo_id == config.hf_canonical_repo and record_import_id == config.raw_import_id and video_id:
            latest_by_video[str(video_id)] = record
    latest_records = list(latest_by_video.values())
    pass_records = [record for record in latest_records if record.get("status") == "pass"]
    failed_records = [record for record in latest_records if record.get("status") == "failed"]
    invalid_timeline_records = [record for record in latest_records if (record.get("inventory") or {}).get("frame_timeline_status") != "pass"]
    print("- progress_records_all_lines:", len(records))
    print("- current_latest_records:", len(latest_records))
    print("- pass_records:", len(pass_records))
    print("- failed_records:", len(failed_records))
    print("- invalid_timeline_records:", len(invalid_timeline_records))
    if failed_records:
        raise RuntimeError(f"Stream upload hiện tại còn failed records, ví dụ: {failed_records[:3]}")
    if not pass_records:
        raise RuntimeError("Stream upload chưa có pair nào status=pass.")
    if config.frame_timeline_policy == "required" and invalid_timeline_records:
        raise RuntimeError(f"Stream progress còn video thiếu timeline pass: {invalid_timeline_records[:3]}")
    leftover = sorted(scratch_dir.glob("stream_pair_*")) if scratch_dir.exists() else []
    print("- scratch leftovers:", leftover[:10])
    if leftover:
        raise RuntimeError(f"Scratch còn stream_pair_* sau cleanup: {leftover[:10]}")
else:
    print("config.run_upload_standardized_raw=False, skip stream progress check.")


In [ ]:
# BƯỚC 10: Kiểm tra HF raw repo sau canonical raw upload.
# Gate trước khi chạy HF ingest: raw repo phải có video/metadata + 5 canonical raw manifests.
# Không hardcode số lượng file; kiểm tra theo trạng thái thực tế trên HF raw repo và import report.

from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import os
import json
import pandas as pd

repo_id = config.hf_canonical_repo
raw_import_id = config.raw_import_id.strip("/")
prefix = raw_import_id + "/"
token = os.environ.get("HF_TOKEN") or os.environ.get("AIC_HF_TOKEN")
if not token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF raw repo.")

api = HfApi(token=token)
files = {entry.path for entry in api.list_repo_tree(repo_id=repo_id, repo_type="dataset", path_in_repo=raw_import_id, recursive=True, token=token) if getattr(entry, "path", None)}

raw_videos = sorted(f for f in files if f.startswith(prefix + "raw_videos/"))
metadata = sorted(f for f in files if f.startswith(prefix + "metadata/"))
frame_timeline = sorted(f for f in files if f.startswith(prefix + "frame_timeline/") and f.endswith(".parquet"))

required_manifests = [
    prefix + "manifests/canonical_file_manifest.jsonl",
    prefix + "manifests/canonical_import_report.json",
    prefix + "manifests/canonical_video_inventory.parquet",
    prefix + "manifests/missing_metadata.json",
    prefix + "manifests/unmatched_metadata.json",
]
missing_manifests = [p for p in required_manifests if p not in files]

print("repo:", repo_id)
print("prefix:", prefix)
print("raw_videos:", len(raw_videos))
print("metadata:", len(metadata))
print("frame_timeline:", len(frame_timeline))
print("required manifests:")
for p in required_manifests:
    print("-", p, "OK" if p in files else "MISSING")

if len(raw_videos) == 0:
    raise RuntimeError("HF raw repo chưa có raw_videos. Chưa chạy canonical raw upload hoặc upload chưa hoàn tất.")

if len(metadata) == 0:
    raise RuntimeError("HF raw repo chưa có metadata. Chưa chạy canonical raw upload hoặc upload chưa hoàn tất.")

if missing_manifests:
    raise RuntimeError(f"HF raw repo thiếu required canonical manifests: {missing_manifests}")

# Package tạo canonical metadata schema 1.0 cho mọi video, kể cả khi organizer metadata bị thiếu.
# Vì vậy raw_videos và metadata trong HF raw repo phải là video-primary pairs.
if len(raw_videos) != len(metadata) or len(raw_videos) != len(frame_timeline):
    raise RuntimeError(
        "HF raw repo có số raw_videos và metadata không khớp. "
        f"raw_videos={len(raw_videos)}, metadata={len(metadata)}, frame_timeline={len(frame_timeline)}. "
        "Cần kiểm tra lại canonical raw upload/canonical manifest."
    )

report_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=prefix + "manifests/canonical_import_report.json",
    token=token,
)
report = json.loads(Path(report_path).read_text(encoding="utf-8"))

print("\ncanonical_import_report:")
print("- status:", report.get("status"))
print("- video_count:", report.get("video_count"))
print("- metadata_count:", report.get("metadata_count"))
print("- frame_timeline_count:", report.get("frame_timeline_count"))
print("- frame_timeline_policy:", report.get("frame_timeline_policy"))
print("- frame_timeline_status_counts:", report.get("frame_timeline_status_counts"))
print("- metadata_schema_version:", report.get("metadata_schema_version"))
print("- organizer_metadata_present_count:", report.get("organizer_metadata_present_count"))
print("- metadata_generated_count:", report.get("metadata_generated_count"))
print("- probe_status_counts:", report.get("probe_status_counts"))
print("- uploaded_pair_count:", report.get("uploaded_pair_count"))
print("- error_count:", report.get("error_count"))
print("- inventory_path:", report.get("inventory_path"))
print("- missing_metadata_path:", report.get("missing_metadata_path"))
print("- unmatched_metadata_path:", report.get("unmatched_metadata_path"))

if report.get("metadata_schema_version") != "1.0":
    raise RuntimeError(f"Raw prefix không dùng canonical metadata schema 1.0: {report.get('metadata_schema_version')}")

uploaded_pair_count = report.get("uploaded_pair_count")
if uploaded_pair_count is not None:
    uploaded_pair_count = int(uploaded_pair_count)
    if len(raw_videos) != uploaded_pair_count or len(metadata) != uploaded_pair_count or len(frame_timeline) != uploaded_pair_count:
        raise RuntimeError(
            "HF raw file count không khớp uploaded_pair_count trong canonical_import_report. "
            f"raw_videos={len(raw_videos)}, metadata={len(metadata)}, frame_timeline={len(frame_timeline)}, uploaded_pair_count={uploaded_pair_count}."
        )

inventory_local = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=prefix + "manifests/canonical_video_inventory.parquet",
    token=token,
)
inventory_df = pd.read_parquet(inventory_local)
required_inventory_cols = {
    "video_id",
    "video_filename",
    "metadata_filename",
    "video_size_bytes",
    "metadata_size_bytes",
    "frame_timeline_size_bytes",
    "canonical_backend",
    "canonical_repo_id",
    "canonical_repo_type",
    "canonical_revision",
    "canonical_prefix",
    "canonical_video_path",
    "canonical_metadata_path",
    "canonical_frame_timeline_path",
    "frame_timeline_status",
    "frame_timeline_row_count",
    "metadata_schema_version",
    "organizer_metadata_present",
    "metadata_generated",
    "duration_sec",
    "fps",
    "frame_count",
    "width",
    "height",
    "is_vfr",
    "file_size_bytes",
    "probe_status",
    "probe_attempts",
}
missing_inventory_cols = sorted(required_inventory_cols - set(inventory_df.columns))
print("\ncanonical_video_inventory:")
print("- rows:", len(inventory_df))
print("- missing_inventory_cols:", missing_inventory_cols)
display(inventory_df.head())

if missing_inventory_cols:
    raise RuntimeError(f"canonical_video_inventory.parquet thiếu cột required: {missing_inventory_cols}")

if len(inventory_df) != len(raw_videos):
    raise RuntimeError(
        "canonical_video_inventory row count không khớp raw_videos. "
        f"inventory_rows={len(inventory_df)}, raw_videos={len(raw_videos)}."
    )

if set(inventory_df["metadata_schema_version"].dropna().astype(str)) != {"1.0"}:
    raise RuntimeError("Inventory chứa metadata schema khác 1.0. Hãy dùng raw_import_id mới.")
if set(inventory_df["frame_timeline_status"].dropna().astype(str)) != {"pass"}:
    raise RuntimeError("Inventory có frame timeline không ở status=pass. Production ingest không được tiếp tục.")
if (pd.to_numeric(inventory_df["frame_timeline_row_count"], errors="coerce").fillna(0) < 1).any():
    raise RuntimeError("Inventory có frame_timeline_row_count < 1.")
if report.get("frame_timeline_policy") != config.frame_timeline_policy:
    raise RuntimeError(f"Raw report timeline policy mismatch: {report.get('frame_timeline_policy')}")

organizer_metadata_files = [f for f in files if f.startswith(prefix + "organizer_metadata/")]
if organizer_metadata_files:
    raise RuntimeError(f"HF raw prefix không được chứa organizer_metadata/: {organizer_metadata_files[:5]}")

for audit_name in ["missing_metadata.json", "unmatched_metadata.json"]:
    audit_local = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=prefix + f"manifests/{audit_name}",
        token=token,
    )
    audit_payload = json.loads(Path(audit_local).read_text(encoding="utf-8"))
    print(f"\n{audit_name}:")
    print("- kind:", audit_payload.get("kind"))
    print("- count:", audit_payload.get("count"))

if int(report.get("error_count", 0) or 0) != 0:
    raise RuntimeError("canonical_import_report còn error_count > 0. Hãy kiểm tra report và rerun canonical raw upload.")

print("\nHF raw repo versioned structure OK. Có thể chạy ingest từ HF.")


In [ ]:
# BƯỚC 11: Ingest từ HF raw repo để tạo processed artifacts local.
# Lưu ý: đây là HF canonical ingest, không phải local ingest từ Drive.
from pathlib import Path
import os
import shutil
import subprocess

release_root = output_base / config.release_id
canonical_staging_root = Path("/content/canonical_staging")

print("BƯỚC 11: Ingest từ HF raw repo")
print("- hf_canonical_repo:", config.hf_canonical_repo)
print("- raw_import_id:", config.raw_import_id)
print("- canonical_staging_root:", canonical_staging_root)
print("- release_root:", release_root)

# Vì command đang chạy --no-resume, xóa staging cũ để tránh trộn state cũ.
# Nếu package sau này có checkpoint/resume thật cho HF ingest, có thể đổi policy này.
if config.run_ingest and canonical_staging_root.exists():
    shutil.rmtree(canonical_staging_root, ignore_errors=True)
if config.run_ingest:
    canonical_staging_root.mkdir(parents=True, exist_ok=True)

print("Disk/cache trước HF ingest:")
subprocess.run("df -h / /content 2>/dev/null || df -h / /content", shell=True, check=False)
subprocess.run(f"du -sh {str(canonical_staging_root)!r} 2>/dev/null || true", shell=True, check=False)
subprocess.run("du -sh /root/.cache/huggingface 2>/dev/null || true", shell=True, check=False)

INGEST_MAX_WORKERS = str(config.ingest_max_workers)
print("- ingest max workers:", INGEST_MAX_WORKERS)
print("- frame timeline policy:", config.frame_timeline_policy)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

ingest_cmd = [
    "ingest",
    "--mode", config.package_mode,
    "--output", str(output_base),
    "--canonical-hf-repo-id", config.hf_canonical_repo,
    "--canonical-hf-prefix", config.raw_import_id,
    "--canonical-staging-root", str(canonical_staging_root),
    "--max-workers", INGEST_MAX_WORKERS,
    "--frame-timeline-policy", config.frame_timeline_policy,
    "--no-resume",
]
if config.run_ingest:
    run_cli(ingest_cmd, stream=False, tail_lines=300)
else:
    print("Bỏ qua ingest theo config; kiểm tra artifact hiện có.")

videos_found = [release_root / "tables" / "videos.parquet"] if (release_root / "tables" / "videos.parquet").exists() else []
media_manifest_found = [release_root / "raw_mapping" / "media_store_manifest.parquet"] if (release_root / "raw_mapping" / "media_store_manifest.parquet").exists() else []
frame_timeline_manifest_found = [release_root / "manifests" / "frame_timeline_manifest.parquet"] if (release_root / "manifests" / "frame_timeline_manifest.parquet").exists() else []
frame_timeline_files_found = sorted((release_root / "frame_timeline").glob("*.parquet")) if (release_root / "frame_timeline").exists() else []

print("videos.parquet found:", videos_found)
print("media_store_manifest.parquet found:", media_manifest_found)
print("frame_timeline_manifest.parquet found:", frame_timeline_manifest_found)
print("frame_timeline/*.parquet count:", len(frame_timeline_files_found))

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet sau ingest.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet sau ingest.")
if not frame_timeline_manifest_found:
    raise RuntimeError("Không tìm thấy frame_timeline_manifest.parquet sau ingest.")
timeline_manifest_df = pd.read_parquet(frame_timeline_manifest_found[0])
if len(frame_timeline_files_found) != len(timeline_manifest_df) or set(timeline_manifest_df["status"].astype(str)) != {"pass"}:
    raise RuntimeError("Phase00 local chưa có timeline status=pass cho mọi video.")

In [ ]:
# BƯỚC 12: Chia batch từ videos.parquet đã tạo sau ingest.
# Lưu ý: workflow hiện tại tạo videos.parquet từ HF raw ingest.
# Command assign-batches hiện tại tự đọc từ output_base theo release; package_mode chỉ giữ tương thích CLI.

print("BƯỚC 12: Assign batches")
print("- output_base:", output_base)
print("- package_mode:", config.package_mode)
print("- num_batches:", config.num_batches)

videos_found = [release_root / "tables" / "videos.parquet"] if (release_root / "tables" / "videos.parquet").exists() else []
media_manifest_found = [release_root / "raw_mapping" / "media_store_manifest.parquet"] if (release_root / "raw_mapping" / "media_store_manifest.parquet").exists() else []
frame_timeline_manifest_found = [release_root / "manifests" / "frame_timeline_manifest.parquet"] if (release_root / "manifests" / "frame_timeline_manifest.parquet").exists() else []

print("videos.parquet found:", videos_found)
print("media_store_manifest.parquet found:", media_manifest_found)
print("frame_timeline_manifest.parquet found:", frame_timeline_manifest_found)

if not videos_found:
    raise RuntimeError("Chưa có videos.parquet. Hãy chạy ingest thành công trước khi assign batches.")
if not frame_timeline_manifest_found:
    raise RuntimeError("Chưa có frame_timeline_manifest.parquet. Hãy chạy ingest thành công trước khi assign batches.")

if config.run_assign_batches:
    run_cli([
        "assign-batches",
        "--mode", config.package_mode,
        "--num-batches", str(config.num_batches),
        "--output", str(output_base),
        "--no-resume",
    ], stream=False, tail_lines=300)
else:
    print("Bỏ qua assign-batches theo config.")

batch_manifest_found = [release_root / "manifests" / "batch_manifest.csv"] if (release_root / "manifests" / "batch_manifest.csv").exists() else []
batch_txt_found = sorted((release_root / "manifests").glob("batch_*.txt"))
video_count = len(pd.read_parquet(videos_found[0]))
expected_batch_count = min(config.num_batches, video_count)

print("batch_manifest.csv found:", batch_manifest_found)
print("batch_*.txt found:", batch_txt_found[:30])

if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv sau assign-batches.")

if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt sau assign-batches.")

if len(batch_txt_found) != expected_batch_count:
    raise RuntimeError(
        f"Số batch_*.txt không khớp: expected={expected_batch_count}, actual={len(batch_txt_found)}"
    )

print("Assign batches OK.")

In [ ]:
# BƯỚC 13: Gom audit reports vào local release manifests.
# sync-phase00-ingestion sẽ map report files sang phase00_ingestion/reports/.

from pathlib import Path
import os
import shutil

from huggingface_hub import hf_hub_download

release_root = output_base / config.release_id
manifests_root = release_root / "manifests"
manifests_root.mkdir(parents=True, exist_ok=True)

required_local_reports = []
if config.run_drive_shadow:
    required_local_reports.append(Path(config.workspace) / "drive_shadow_report.json")

if config.run_upload_standardized_raw:
    required_local_reports.append(Path(config.stream_progress_path))

optional_local_reports = []

missing_required_reports = [p for p in required_local_reports if not p.exists()]
if missing_required_reports:
    raise RuntimeError(
        "Thiếu required local audit reports trước khi upload processed artifacts: "
        + ", ".join(str(p) for p in missing_required_reports)
    )

for report_path in required_local_reports + optional_local_reports:
    if report_path.exists():
        target = manifests_root / report_path.name
        shutil.copy2(report_path, target)
        print("Copied local report:", report_path, "->", target)
    else:
        print("Optional local report not found, skip:", report_path)

# Snapshot raw-level canonical import report từ AIC26_raw sang phase00 reports.
token = os.environ.get("HF_TOKEN") or os.environ.get("AIC_HF_TOKEN")
if not token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để snapshot raw canonical reports.")

raw_prefix = config.raw_import_id.strip("/")
raw_reports = [
    "canonical_import_report.json",
]
for report_name in raw_reports:
    local_path = hf_hub_download(
        repo_id=config.hf_canonical_repo,
        repo_type="dataset",
        filename=f"{raw_prefix}/manifests/{report_name}",
        token=token,
    )
    target = manifests_root / report_name
    shutil.copy2(local_path, target)
    print("Copied raw HF report:", f"{raw_prefix}/manifests/{report_name}", "->", target)

print("manifests:")
for p in sorted(manifests_root.iterdir()):
    print(p)

print("Release root:", release_root)
print("Release files:")
for p in sorted(release_root.rglob("*"))[:100]:
    print(" ", p)


In [ ]:
# BƯỚC 14: Upload phase00 ingestion artifacts lên Hugging Face processed repo.
# Không upload raw_videos/metadata ở bước này.
# Raw media thật đã nằm trong HF raw repo. Processed repo chỉ nhận phase00 tables/raw_mapping/manifests/reports.

from pathlib import Path
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

release_root = output_base / config.release_id
processed_repo_id = config.hf_release_repo
phase00_remote_root = f"{config.release_id}/phase00_ingestion"

print("BƯỚC 14: Upload phase00 ingestion artifacts")
print("- run_upload_processed:", config.run_upload_processed)
print("- processed_repo_id:", processed_repo_id)
print("- release_root:", release_root)
print("- phase00_remote_root:", phase00_remote_root)

if not config.run_upload_processed:
    print("Bỏ qua upload phase00 ingestion artifacts theo config.")
else:
    if not release_root.exists():
        raise RuntimeError(f"Không thấy release_root: {release_root}")

    required_local_files = [
        release_root / "tables" / "videos.parquet",
        release_root / "raw_mapping" / "media_store_manifest.parquet",
        release_root / "manifests" / "frame_timeline_manifest.parquet",
        release_root / "manifests" / "batch_manifest.csv",
        release_root / "manifests" / "dataset_report.json",
        release_root / "manifests" / "ingestion_errors.jsonl",
        release_root / "manifests" / "missing_metadata.json",
        release_root / "manifests" / "unmatched_metadata.json",
    ]

    if config.run_drive_shadow:
        required_local_files.append(release_root / "manifests" / "drive_shadow_report.json")

    if config.run_upload_standardized_raw:
        required_local_files.extend([
            release_root / "manifests" / "canonical_import_report.json",
            release_root / "manifests" / Path(config.stream_progress_path).name,
        ])

    missing_local = [p for p in required_local_files if not p.exists()]
    if missing_local:
        raise RuntimeError(
            "Thiếu local phase00 files trước khi sync lên HF: "
            + ", ".join(str(p) for p in missing_local)
        )

    batch_txt_files = sorted((release_root / "manifests").glob("batch_*.txt"))
    if not batch_txt_files:
        raise RuntimeError(f"Không thấy batch_*.txt trong {release_root / 'manifests'}")

    print("Local phase00 files ready:")
    for p in required_local_files:
        print("-", p)
    print("- batch_*.txt count:", len(batch_txt_files))

    run_cli([
        "sync-phase00-ingestion",
        "--output", str(output_base),
        "--hf-repo-id", processed_repo_id,
    ], stream=False, tail_lines=300)

    print("Uploaded phase00 ingestion artifacts to HF:", processed_repo_id)


In [ ]:
# BƯỚC 15: Kiểm tra cấu trúc HF processed repo theo phase00_ingestion layout.
# Layout mới:
# AIC26_release/<release_id>/phase00_ingestion/{tables,raw_mapping,frame_timeline,manifests,reports}
# Legacy flat layout <release_id>/{tables,raw_mapping,manifests} chỉ được coi là deprecated.

from pathlib import Path
import os
import pandas as pd

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi, hf_hub_download

repo_id = config.hf_release_repo
release_id = config.release_id
phase00_root = f"{release_id}/phase00_ingestion"

hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
            os.environ["AIC_HF_TOKEN"] = hf_token
    except Exception:
        pass

if not hf_token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF repo.")

api = HfApi(token=hf_token)
files = [entry.path for entry in api.list_repo_tree(repo_id=repo_id, repo_type="dataset", path_in_repo=release_id, recursive=True, token=hf_token) if getattr(entry, "path", None)]

release_files = sorted([f for f in files if f.startswith(f"{release_id}/")])
phase00_files = sorted([f for f in files if f.startswith(f"{phase00_root}/")])

print("HF processed repo:", repo_id)
print("release_id:", release_id)
print("phase00_root:", phase00_root)
print("release_file_count:", len(release_files))
print("phase00_file_count:", len(phase00_files))

for f in phase00_files:
    print(" ", f)

required_exact = [
    f"{phase00_root}/tables/videos.parquet",
    f"{phase00_root}/raw_mapping/media_store_manifest.parquet",
    f"{phase00_root}/manifests/frame_timeline_manifest.parquet",
    f"{phase00_root}/reports/dataset_report.json",
    f"{phase00_root}/reports/ingestion_errors.jsonl",
    f"{phase00_root}/reports/missing_metadata.json",
    f"{phase00_root}/reports/unmatched_metadata.json",
    f"{phase00_root}/manifests/batch_manifest.csv",
    f"{phase00_root}/reports/phase00_sync_manifest.json",
]

if config.run_drive_shadow:
    required_exact.append(f"{phase00_root}/reports/drive_shadow_report.json")

if config.run_upload_standardized_raw:
    required_exact.extend([
        f"{phase00_root}/reports/canonical_import_report.json",
        f"{phase00_root}/reports/{Path(config.stream_progress_path).name}",
    ])

missing_required = [p for p in required_exact if p not in phase00_files]

frame_timeline_files = sorted([
    p for p in phase00_files
    if p.startswith(f"{phase00_root}/frame_timeline/") and p.endswith(".parquet")
])

batch_txt_files = sorted([
    p for p in phase00_files
    if p.startswith(f"{phase00_root}/manifests/batch_") and p.endswith(".txt")
])

forbidden_patterns = [
    "/raw_videos/",
    "/metadata/",
    "/temp_extract/",
    "member_stage_",
    "member_extract_",
]

forbidden_files = [
    p for p in release_files
    if any(pattern in p for pattern in forbidden_patterns)
    or p.lower().endswith((".mp4", ".mov", ".mkv", ".avi", ".webm", ".wav", ".zip"))
]

legacy_flat_files = sorted([
    p for p in release_files
    if p.startswith(f"{release_id}/tables/")
    or p.startswith(f"{release_id}/raw_mapping/")
    or p.startswith(f"{release_id}/manifests/")
])

print("\nCHECK REQUIRED:")
print("- missing_required:", missing_required)
print("- batch_txt_count:", len(batch_txt_files))
print("- batch_txt_files:", batch_txt_files)
print("- frame_timeline_file_count:", len(frame_timeline_files))

print("\nCHECK FORBIDDEN:")
print("- forbidden_files:", forbidden_files)

print("\nCHECK LEGACY FLAT LAYOUT:")
print("- legacy_flat_files_count:", len(legacy_flat_files))
if legacy_flat_files:
    print("WARNING: Repo còn legacy flat layout đã deprecated. Không fail để giữ backward compatibility, nhưng output mới nên dùng phase00_ingestion.")
    for p in legacy_flat_files[:50]:
        print(" legacy:", p)

if missing_required:
    raise RuntimeError(f"Thiếu required files trên HF processed repo phase00_ingestion: {missing_required}")

if len(batch_txt_files) == 0:
    raise RuntimeError("Không thấy phase00_ingestion/manifests/batch_*.txt trên HF processed repo.")

if forbidden_files:
    raise RuntimeError(f"HF processed repo có file không nên upload: {forbidden_files}")

# Kiểm tra nội dung nếu raw upload đã bật:
# media_store_manifest phải có các cột canonical cần thiết.
# Với raw version/prefix, chấp nhận một trong hai:
# - canonical_import_id: tên cũ/notebook cũ
# - canonical_prefix: tên hiện tại trong package
if config.run_upload_standardized_raw:
    frame_timeline_manifest_local = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=f"{phase00_root}/manifests/frame_timeline_manifest.parquet",
        token=hf_token,
    )
    frame_timeline_manifest_df = pd.read_parquet(frame_timeline_manifest_local)
    print("\nCHECK FRAME TIMELINE MANIFEST:")
    print("- rows:", len(frame_timeline_manifest_df))
    if "status" in frame_timeline_manifest_df.columns:
        print("- status counts:", frame_timeline_manifest_df["status"].value_counts(dropna=False).to_dict())
    if len(frame_timeline_files) != len(frame_timeline_manifest_df) or set(frame_timeline_manifest_df["status"].astype(str)) != {"pass"}:
        raise RuntimeError("HF Phase00 chưa có timeline status=pass cho mọi video.")

    media_manifest_local = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=f"{phase00_root}/raw_mapping/media_store_manifest.parquet",
        token=hf_token,
    )

    media_df = pd.read_parquet(media_manifest_local)

    required_canonical_cols = {
        "canonical_backend",
        "canonical_repo_id",
        "canonical_repo_type",
        "canonical_video_path",
        "canonical_metadata_path",
        "canonical_frame_timeline_path",
    }

    missing_canonical_cols = sorted(required_canonical_cols - set(media_df.columns))

    has_import_id = "canonical_import_id" in media_df.columns
    has_prefix = "canonical_prefix" in media_df.columns

    print("\nCHECK CANONICAL COLUMNS:")
    print("- columns:", list(media_df.columns))
    print("- missing_canonical_cols:", missing_canonical_cols)
    print("- has canonical_import_id:", has_import_id)
    print("- has canonical_prefix:", has_prefix)

    if missing_canonical_cols:
        raise RuntimeError(f"media_store_manifest thiếu canonical columns: {missing_canonical_cols}")

    if not (has_import_id or has_prefix):
        raise RuntimeError(
            "media_store_manifest thiếu cả canonical_import_id và canonical_prefix. "
            "Cần có ít nhất một cột để xác định HF raw prefix/version."
        )

    if "canonical_import_id" not in media_df.columns and "canonical_prefix" in media_df.columns:
        media_df["canonical_import_id"] = media_df["canonical_prefix"]

    print("- canonical prefix/import sample:")
    display(media_df[[
        "video_id",
        "canonical_repo_id",
        "canonical_repo_type",
        "canonical_import_id",
        "canonical_video_path",
        "canonical_metadata_path",
    ]].head())

print("\nHF processed phase00_ingestion structure OK.")


In [ ]:
# BƯỚC 16: Preview output của notebook 00.

from pathlib import Path
import pandas as pd

release_root = output_base / config.release_id
videos_found = [release_root / "tables" / "videos.parquet"] if (release_root / "tables" / "videos.parquet").exists() else []
batch_manifest_found = [release_root / "manifests" / "batch_manifest.csv"] if (release_root / "manifests" / "batch_manifest.csv").exists() else []
media_manifest_found = [release_root / "raw_mapping" / "media_store_manifest.parquet"] if (release_root / "raw_mapping" / "media_store_manifest.parquet").exists() else []
frame_timeline_manifest_found = [release_root / "manifests" / "frame_timeline_manifest.parquet"] if (release_root / "manifests" / "frame_timeline_manifest.parquet").exists() else []
frame_timeline_files_found = sorted((release_root / "frame_timeline").glob("*.parquet")) if (release_root / "frame_timeline").exists() else []
batch_txt_found = sorted((release_root / "manifests").glob("batch_*.txt"))

print("Preview notebook 00 outputs")
print("- videos.parquet found:", videos_found)
print("- media_store_manifest.parquet found:", media_manifest_found)
print("- frame_timeline_manifest.parquet found:", frame_timeline_manifest_found)
print("- frame_timeline/*.parquet count:", len(frame_timeline_files_found))
print("- batch_manifest.csv found:", batch_manifest_found)
print("- batch_*.txt count:", len(batch_txt_found))

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet.")
if not frame_timeline_manifest_found:
    raise RuntimeError("Không tìm thấy frame_timeline_manifest.parquet.")
if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv.")
if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt.")

videos_path = videos_found[0]
media_manifest_path = media_manifest_found[0]
batch_manifest_path = batch_manifest_found[0]
frame_timeline_manifest_path = frame_timeline_manifest_found[0]

videos_df = pd.read_parquet(videos_path)
media_df = pd.read_parquet(media_manifest_path)
batch_df = pd.read_csv(batch_manifest_path)
frame_timeline_manifest_df = pd.read_parquet(frame_timeline_manifest_path)

print("\nvideos.parquet:", videos_path)
print("video_count:", len(videos_df))
display(videos_df.head())

print("\nmedia_store_manifest.parquet:", media_manifest_path)
print("media_manifest_rows:", len(media_df))
display(media_df.head())

print("\nframe_timeline_manifest.parquet:", frame_timeline_manifest_path)
print("frame_timeline_manifest_rows:", len(frame_timeline_manifest_df))
if "status" in frame_timeline_manifest_df.columns:
    print("frame_timeline_status_counts:", frame_timeline_manifest_df["status"].value_counts(dropna=False).to_dict())
display(frame_timeline_manifest_df.head())

print("\nbatch_manifest.csv:", batch_manifest_path)
print("batch_rows:", len(batch_df))
display(batch_df.head())

print("\nBatch txt files:")
for p in batch_txt_found[:30]:
    print(" ", p)

print("\nNotebook 00 hoàn tất nếu cell này chạy xong.")

In [ ]:
# BƯỚC 17 OPTIONAL: Kiểm tra lại HF raw repo versioned sau toàn bộ notebook.
import os

try:
    from huggingface_hub import HfApi
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

raw_repo_id = config.hf_canonical_repo
raw_import_id = config.raw_import_id.strip("/")

print("BƯỚC 17 OPTIONAL: Check HF raw repo")
print("- run_upload_standardized_raw:", config.run_upload_standardized_raw)
print("- raw_repo_id:", raw_repo_id)
print("- raw_import_id:", raw_import_id)

if not config.run_upload_standardized_raw:
    print("config.run_upload_standardized_raw=False, skip HF raw repo check.")
else:
    hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                os.environ["HF_TOKEN"] = hf_token
                os.environ["AIC_HF_TOKEN"] = hf_token
        except Exception:
            pass
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF raw repo.")

    api = HfApi(token=hf_token)
    files = sorted(entry.path for entry in api.list_repo_tree(repo_id=raw_repo_id, repo_type="dataset", path_in_repo=raw_import_id, recursive=True, token=hf_token) if getattr(entry, "path", None))

    prefix = f"{raw_import_id}/"
    prefix_files = [f for f in files if f.startswith(prefix)]

    raw_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/raw_videos/")]
    metadata_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/metadata/")]
    timeline_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/frame_timeline/") and f.endswith(".parquet")]
    manifest_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/manifests/")]

    print("prefix_file_count:", len(prefix_files))
    print("raw_videos count:", len(raw_files))
    print("metadata count:", len(metadata_files))
    print("frame_timeline count:", len(timeline_files))
    print("manifests:")
    for f in manifest_files:
        print(" ", f)

    required = [
        f"{raw_import_id}/manifests/canonical_file_manifest.jsonl",
        f"{raw_import_id}/manifests/canonical_import_report.json",
        f"{raw_import_id}/manifests/canonical_video_inventory.parquet",
        f"{raw_import_id}/manifests/missing_metadata.json",
        f"{raw_import_id}/manifests/unmatched_metadata.json",
    ]

    missing = [p for p in required if p not in files]

    forbidden = [
        f for f in prefix_files
        if "standardize_progress.jsonl" in f
        or "standardize_archives_report.json" in f
        or "drive_shadow_report.json" in f
        or "batch_" in f
        or f.endswith("videos.parquet")
        or f.endswith("media_store_manifest.parquet")
    ]

    print("missing required:", missing)
    print("forbidden files:", forbidden)

    if not raw_files:
        raise RuntimeError("HF raw repo không có raw_videos.")
    if not metadata_files:
        raise RuntimeError("HF raw repo không có metadata.")
    if len(raw_files) != len(metadata_files) or len(raw_files) != len(timeline_files):
        raise RuntimeError(f"HF raw pair count mismatch: videos={len(raw_files)}, metadata={len(metadata_files)}, timeline={len(timeline_files)}")
    if missing:
        raise RuntimeError(f"HF raw repo thiếu required manifests: {missing}")
    if forbidden:
        raise RuntimeError(f"HF raw repo có file không đúng mục đích: {forbidden}")

    print("HF raw repo versioned structure OK.")
